<a href="https://colab.research.google.com/github/animesh-kishore/llama3.2_3b_4bits_qlora/blob/main/llama3_2_3b_4bits_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

In [8]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

# load quantized llama3.2 3B

# Set bnb config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # load llama3.2 3B in symmetric 4bits
    bnb_4bit_quant_type="nf4", # Quantization dtype
    bnb_4bit_use_double_quant=True, # Quantize scale as well. Saves more memory
    bnb_4bit_compute_dtype=torch.float16, # Quantized parameters are dequantized in this dtype before calculation
)

# Set tokenzier
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Used to make all sentences in a batch of same size. llama3.2 doesn't implicitly provide pad_token
tokenizer.padding_side = "right" # Add padding after end of sentences

# Set model
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B",
    quantization_config=bnb_config,
    device_map="auto", # Automatically select underlying HW for execution
)
base_model.generation_config.pad_token_id = tokenizer.pad_token # llama3.2 doesn't have pad tokens implicitly set.

print(base_model)
print(f"Memory footprint {base_model.get_memory_footprint() / 1e6} MB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [9]:
from peft import LoraConfig # peft is parameter efficient fine tuning

LORA_R = 32
LORA_ALPHA = 2 * LORA_R # Typically 2 * r
LORA_DROPOUT = 0.1 # zero out 10% of lora inputs on an average
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"] + ["gate_proj", "up_proj", "down_proj"] # Attention + MLP

# Create loRA config
lora_params = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none", # Do not update base model bias
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES
)

In [10]:
from trl import SFTConfig # Transformer Reinforcement Learning used to refine LLMs post pre-training
from datetime import datetime

EPOCHS = 1
BATCH_SIZE = 32
OPTIMIZER = "paged_adamw_32bit"
MAX_SEQUENCE_LENGTH = 128
HUB_MODEL_ID="animeshkishore/llama3.2_3b_4bits_qlora"

# Create training config for Supervised Fine Tuning
train_params = SFTConfig(
    output_dir="sft_output", # local working directory
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, # per device (e.g. GPU, CPU) number of prompt-completion pairs that will be processed simultaneously
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1, # Accumulate gradient for multiple batches and update weights in one shot
    optim=OPTIMIZER,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.001,
    fp16=True, # Lora weights are still saved in fp32. However, during training it's casted to fp16 and used. Should be same as bnb_4bit_compute_dtype.
    max_grad_norm=0.3,
    warmup_ratio=0.01,
    group_by_length=True, # Group sequences of similar length in a batch together. Avoid memory wastage due to padding.
    max_length=MAX_SEQUENCE_LENGTH, # Max tokens per input sequence in a batch. Anything smaller is padded. Larger is chopped.
    save_steps=100, # saves checkpoint at output_dir
    save_total_limit=1, # Save only 1 checkpoint locally. Rest all will be saved on HF.
    save_strategy="steps",
    push_to_hub=True, # push checkpoints to HF as well
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="every_save", # push to HF for every local checkpoint save
    eval_strategy="steps",
    eval_steps=100, # Specify how often evaluation is run on validation dataset
    logging_steps=5, # Frequency at which training stats are send to wandb.ai
    report_to="wandb", # Send training stats to wandb.ai
    run_name=f"llama3.2_3b_4bits_qlora-{datetime.now():%Y-%m-%d_%H.%M.%S}", # Set run name to view on wandb.ai
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
from trl import SFTTrainer
from datasets import load_dataset

DATASET_NAME = "ed-donner/items_prompts_lite"
dataset = load_dataset(DATASET_NAME) # load fine tuning dataset

# Create SFT trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"].select(range(500)),
    peft_config=lora_params,
    args=train_params
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [13]:
# Start training

fine_tuning.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: animesh-kishore (models-self2694) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.300503,1.284047,2.670327,330223.000000,0.759500
200,1.263151,1.343051,2.707097,659017.000000,0.758000
300,1.266738,1.275234,2.754432,988895.000000,0.761500
400,1.248553,1.250833,2.741238,1319755.000000,0.762000
500,1.227405,1.248318,2.749860,1649501.000000,0.759500
600,1.264247,1.240087,2.747391,1978536.000000,0.760000


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=625, training_loss=1.2778524063110352, metrics={'train_runtime': 4693.4235, 'train_samples_per_second': 4.261, 'train_steps_per_second': 0.133, 'total_flos': 3.565766747730739e+16, 'train_loss': 1.2778524063110352, 'epoch': 1.0})